# 02 - RAG Generation with HF Wikipedia

This notebook runs headline-only RAG using the embedded Wikipedia dataset from Hugging Face (`not-lain/wikipedia`, revision `embedded`). It evaluates Llama, Qwen, and Mistral with Wikipedia corpus sizes from 5k to 25k documents. Word-pair examples are generated without retrieved context.

**Nota sul kernel**

Seleziona il kernel `LLM-Project (conda)` nel kernel picker in alto a destra prima di eseguire le celle. (Esempio: apri il kernel picker → scegli "LLM-Project (conda)")


## Shared settings

The complete matrix contains 15 runs: three models multiplied by five Wikipedia corpus sizes. `K` is the number of retrieved contexts passed to the generation model. Existing outputs are skipped by default, so an interrupted batch can be resumed.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

MODELS = ("llama", "qwen", "mistral")
N_DOC_COUNTS = tuple(range(5_000, 25_001, 5_000))
DATA = "data/raw/task-a-en.tsv"
RAG_CONFIG = "configs/rag.yaml"
K = 4
APPLY_TO = "headline"
SKIP_EXISTING = False


def _project_root() -> Path:
    root = Path.cwd().resolve()
    if (root / "scripts" / "run_rag_generate.py").exists():
        return root
    candidate = root.parent
    if (candidate / "scripts" / "run_rag_generate.py").exists():
        return candidate
    raise RuntimeError(f"Could not locate the repository root from {root}")


def _use_mock_run(mock: bool | None = None) -> bool:
    if mock is not None:
        return mock
    try:
        import torch
    except ImportError:
        return True
    return not torch.cuda.is_available()


def run_rag_matrix(output_suffix="", mock: bool | None = None):
    root = _project_root()
    use_mock = _use_mock_run(mock)
    total = len(MODELS) * len(N_DOC_COUNTS)
    completed = 0

    for model in MODELS:
        for n_docs in N_DOC_COUNTS:
            if model == "llama" and not os.getenv("HF_TOKEN"):
                completed += 1
                print(f"[{completed}/{total}] Skipping model={model}, documents={n_docs:,} (HF_TOKEN missing).")
                continue

            output_path = root / Path(
                f"data/generated/rag/{model}_rag_wiki_headline_"
                f"{n_docs // 1000}k_k{K}{output_suffix}.jsonl"
            )
            if SKIP_EXISTING and output_path.exists():
                completed += 1
                print(f"[{completed}/{total}] Skipping existing output: {output_path}")
                continue

            output_path.parent.mkdir(parents=True, exist_ok=True)
            cmd = [
                sys.executable, str(root / "scripts" / "run_rag_generate.py"),
                "--model", model,
                "--input", str(root / DATA),
                "--output", str(output_path),
                "--rag-config", str(root / RAG_CONFIG),
                "--n-docs", str(n_docs),
                "--k", str(K),
                "--apply-to", APPLY_TO,
                "--overwrite",
            ]
            if use_mock:
                cmd.append("--mock")
            print(f"[{completed + 1}/{total}] model={model}, documents={n_docs:,}, mode={'mock' if use_mock else 'real'}")
            subprocess.run(cmd, check=True, cwd=root)
            completed += 1
            print(f"Saved: {output_path}")

    print(f"Matrix complete: {completed}/{total} outputs available.")

print(f"Models: {', '.join(MODELS)}")
print(f"Wikipedia corpus sizes: {[f'{n // 1000}k' for n in N_DOC_COUNTS]}")
print(f"Runs per environment: {len(MODELS) * len(N_DOC_COUNTS)}")

Models: llama, qwen, mistral
Wikipedia corpus sizes: ['5k', '10k', '15k', '20k', '25k']
Runs per environment: 15


# Remote execution on Colab

Run this chapter in Google Colab. The repository is expected at `/content/LLM-Project-SemEval-Humor-Generation`. Llama also requires an accepted Hugging Face license and an authenticated Hugging Face account.

In [ ]:
%cd /content/LLM-Project-SemEval-Humor-Generation
%pip install -r requirements-colab.txt

## Run the complete Colab matrix

Set `RUN_COLAB_MATRIX` to `True` to launch all 15 combinations. Colab outputs do not have an environment suffix.

In [ ]:
RUN_COLAB_MATRIX = False

if RUN_COLAB_MATRIX:
    run_rag_matrix()
else:
    print("Set RUN_COLAB_MATRIX = True to launch all 15 Colab runs.")

## Check Colab outputs

In [ ]:
for model in MODELS:
    for n_docs in N_DOC_COUNTS:
        output_path = Path(
            f"data/generated/rag/{model}_rag_wiki_headline_{n_docs // 1000}k_k{K}.jsonl"
        )
        status = "ready" if output_path.exists() else "missing"
        print(f"{status:7}  {output_path}")

# Local execution with the .venv kernel

Run this chapter on a local workstation after selecting the project `.venv` as the Jupyter kernel. The subprocesses use `sys.executable`, so generation runs with that same environment.

In [2]:
def _project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scripts" / "run_rag_generate.py").exists():
            return candidate
    raise RuntimeError(f"Could not locate the repository root from {current}")


ROOT = _project_root()
print(f"Repository root: {ROOT}")
print(f"Python interpreter: {sys.executable}")

Repository root: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation
Python interpreter: c:\Users\peppe\miniconda3\envs\llmproj\python.exe


## Local dependency install

Run this once in the selected `.venv` kernel. It can be skipped when the environment already contains the project dependencies and GPU stack.

In [3]:
%pip install -r requirements-colab.txt

import getpass
import os

os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token: ")
print("Token configurato per questa sessione.")

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-colab.txt'


Token configurato per questa sessione.


## Run the complete local matrix

Set `RUN_LOCAL_MATRIX` to `True` to launch all 15 combinations. Local output names use `_local` to avoid overwriting Colab results.

In [4]:
RUN_LOCAL_MATRIX = True

if RUN_LOCAL_MATRIX:
    run_rag_matrix(output_suffix="_local")
else:
    print("Set RUN_LOCAL_MATRIX = True to launch all 15 local runs.")

[1/15] model=llama, documents=5,000, mode=real
Saved: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_5k_k4_local.jsonl
[2/15] model=llama, documents=10,000, mode=real
Saved: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_10k_k4_local.jsonl
[3/15] model=llama, documents=15,000, mode=real
Saved: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_15k_k4_local.jsonl
[4/15] model=llama, documents=20,000, mode=real
Saved: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_20k_k4_local.jsonl
[5/15] model=llama, documents=25,000, mode=real
Saved: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_25k_k4_local.jsonl
[6/15] model=qwen, documents=5,000, mode=real
Saved: C:\Users\peppe\OneD

## Check local outputs

In [ ]:
for model in MODELS:
    for n_docs in N_DOC_COUNTS:
        #C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data
        output_path = Path(
            f"C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\data/generated/rag/{model}_rag_wiki_headline_{n_docs // 1000}k_k{K}_local.jsonl"
            )
        status = "ready" if output_path.exists() else "missing"
        print(f"{status:7}  {output_path}")

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (3429767386.py, line 6)

In [7]:
import sys
import torch
print("python:", sys.executable)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device_count:", torch.cuda.device_count())
    print("device_name:", torch.cuda.get_device_name(0))


python: c:\Users\peppe\miniconda3\envs\llmproj\python.exe
torch: 2.5.1
cuda_available: True
device_count: 1
device_name: NVIDIA GeForce RTX 4070
